# Cleanup v1.0 Fine-Tuning Artifacts

This notebook archives and quarantines the invalid v1.0 fine-tuning artifacts produced under the normalization defect (FinetuneDataset never z-scored targets at load time, while `evaluate()` de-normalized — applying an inverse transform to un-transformed data). It follows a strict **inventory → copy → verify → move** order and **never deletes anything**.

| Phase | Purpose | Destructive? |
|-------|---------|-------------|
| 0 — Inventory | Read-only audit of every relevant path | No |
| 1 — Verify survivors | Confirm KEEP paths are sound *before* any move | No |
| 2 — Archive | Copy QUARANTINE paths, verify copies, then move originals | Yes (moves only) |
| 3 — Clean-state assertion | Assert workspace is ready for v2.0 Chunk 12 | No |

## 1. Environment Setup & Dependencies

Install the required Python packages for running the notebook.

In [1]:
# Cell 1 — Install dependencies
!pip install scipy numpy matplotlib torch torchvision \
    torch-geometric tqdm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 18.4 MB/s eta 0:00:00


## 2. Repository Setup

Clone the GitHub repository to the local Colab environment and add `src` to the Python path.

In [2]:
# Cell 2 — Clone repo (re-clones every session; pulls latest if already exists)
import os
REPO_ROOT = '/content/antenna-gnn'
if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/asparagusD/antenna_gnn.git {REPO_ROOT}
else:
    !git -C {REPO_ROOT} pull --quiet
import sys
sys.path.insert(0, f'{REPO_ROOT}/src')   # makes 'from model import AntennaGNN' work
print(f'Repo ready at {REPO_ROOT}')

Cloning into '/content/antenna-gnn'...
remote: Enumerating objects: 261, done.
remote: Counting objects: 100% (261/261), done.
remote: Compressing objects: 100% (147/147), done.
remote: Total 261 (delta 137), reused 223 (delta 101), pack-reused 0 (from 0)
Receiving objects: 100% (261/261), 5.42 MiB | 21.01 MiB/s, done.
Resolving deltas: 100% (137/137), done.
Repo ready at /content/antenna-gnn


## 3. Drive Mount and Path Configuration

Mount Google Drive to access `RAW_DATA` and the persistent `DATA_ROOT` workspace.

In [3]:
# Cell 3 — Mount Drive and set data paths
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = '/content/drive/MyDrive/antenna_gnn'
RAW_DATA  = '/content/drive/MyDrive/antenna_dataset'
print(f'Drive mounted. DATA_ROOT={DATA_ROOT}')
print(f'RAW_DATA={RAW_DATA}')

Mounted at /content/drive
Drive mounted. DATA_ROOT=/content/drive/MyDrive/antenna_gnn
RAW_DATA=/content/drive/MyDrive/antenna_dataset


---
# PHASE 0 — Inventory (READ ONLY, changes nothing)

Walk `DATA_ROOT` and classify every relevant path as KEEP, QUARANTINE, or INSPECT.

## CELL A — Full Path Inventory

Print a table of every KEEP / QUARANTINE / INSPECT path with existence, size, and modification time. Summarize how many KEEP paths are missing and how many QUARANTINE paths are present.

In [4]:
# CELL A — Full path inventory (read-only)
import os
import glob
from datetime import datetime

def path_info(base, rel):
    """Return (exists, size_str, mtime_str) for a path relative to base."""
    full = os.path.join(base, rel)
    if not os.path.exists(full):
        return False, '—', '—'
    if os.path.isdir(full):
        pt_files = glob.glob(os.path.join(full, '**', '*.pt'), recursive=True)
        all_files = []
        for root, dirs, files in os.walk(full):
            all_files.extend(files)
        size_str = f'{len(pt_files)} .pt / {len(all_files)} total files'
    else:
        size_bytes = os.path.getsize(full)
        size_str = f'{size_bytes / 1e6:.2f} MB'
    mtime = datetime.fromtimestamp(os.path.getmtime(full)).strftime('%Y-%m-%d %H:%M')
    return True, size_str, mtime

# ── KEEP paths (must exist, must not be touched) ──
KEEP_PATHS = [
    'checkpoints/best_model.pt',
    'artifacts/s11_mean.npy',
    'artifacts/s11_std.npy',
    'artifacts/seed_mask_25.npy',
    'artifacts/seed_mask_35.npy',
    'artifacts/seed_mask_45.npy',
    'artifacts/seed_mask_55.npy',
    'artifacts/finetune_manifest.csv',
    'splits/indices.json',
    'splits/finetune_pool_indices.json',
    'splits/finetune_val_indices.json',
    'splits/finetune_test_indices.json',
    'data/processed_finetune/35x35/',
    'data/processed_finetune/45x45/',
    'data/processed_finetune/55x55/',
]

# ── QUARANTINE paths (invalid, produced under the normalization defect) ──
QUARANTINE_PATHS = [
    'artifacts/transfer_strategy_comparison.csv',
    'artifacts/chosen_transfer_strategy.json',
    'artifacts/al_curves.csv',
    'artifacts/random_results.csv',
    'artifacts/al_composition.csv',
    'artifacts/final_test_results.json',
    'artifacts/label_definition_reconciliation.json',
    'artifacts/al_round_states/',
    'checkpoints/strategy_A.pt',
    'checkpoints/strategy_B.pt',
    'checkpoints/strategy_C.pt',
    'checkpoints/strategy_D.pt',
    'checkpoints/gnn_finetuned_al.pt',
    'checkpoints/gnn_finetuned_random.pt',
    'checkpoints/gnn_finetuned_multiscale.pt',
    'figures/active_learning_curves.png',
    'figures/al_selection_composition.png',
]

# ── INSPECT paths (may or may not exist; report, decide later) ──
INSPECT_PATHS = [
    'artifacts/s11_mean_clean.npy',
    'artifacts/s11_std_clean.npy',
    'splits/train_legacy_independent_split.json',
    'splits/val_legacy_independent_split.json',
    'splits/test_legacy_independent_split.json',
    'data/processed/35x35/',
    'data/processed/45x45/',
    'data/processed/55x55/',
]

# Also check for any pre-existing prenorm_archive/ directories
prenorm_dirs = glob.glob(os.path.join(DATA_ROOT, '**/prenorm_archive*'), recursive=True)
for d in prenorm_dirs:
    rel = os.path.relpath(d, DATA_ROOT)
    if rel not in INSPECT_PATHS:
        INSPECT_PATHS.append(rel)

# ── Print table ──
def print_section(label, paths):
    print(f'\n{"=" * 80}')
    print(f'  {label}')
    print(f'{"=" * 80}')
    print(f'{"Path":<60} {"Exists":<8} {"Size":<30} {"Mtime"}')
    print('-' * 120)
    missing = 0
    present = 0
    for p in paths:
        exists, size, mtime = path_info(DATA_ROOT, p)
        tag = '✓' if exists else '✗'
        print(f'{p:<60} {tag:<8} {size:<30} {mtime}')
        if exists:
            present += 1
        else:
            missing += 1
    return present, missing

keep_present, keep_missing = print_section('KEEP (must exist, must not be touched)', KEEP_PATHS)
q_present, q_missing = print_section('QUARANTINE (invalid v1.0 outputs)', QUARANTINE_PATHS)
i_present, i_missing = print_section('INSPECT (report only, decide later)', INSPECT_PATHS)

print(f'\n{"=" * 80}')
print(f'  SUMMARY')
print(f'{"=" * 80}')
print(f'  KEEP paths missing:      {keep_missing}  (should be 0)')
print(f'  QUARANTINE paths present: {q_present}  (will be archived and moved)')
if keep_missing > 0:
    print('  ⚠ WARNING: Some KEEP paths are missing! Investigate before proceeding.')
print()


  KEEP (must exist, must not be touched)
Path                                                         Exists   Size                           Mtime
------------------------------------------------------------------------------------------------------------------------
checkpoints/best_model.pt                                    ✓        7.15 MB                        2026-07-18 07:23
artifacts/s11_mean.npy                                       ✓        0.00 MB                        2026-07-13 12:16
artifacts/s11_std.npy                                        ✓        0.00 MB                        2026-07-13 12:16
artifacts/seed_mask_25.npy                                   ✓        0.00 MB                        2026-07-11 09:20
artifacts/seed_mask_35.npy                                   ✓        0.00 MB                        2026-07-11 09:21
artifacts/seed_mask_45.npy                                   ✓        0.00 MB                        2026-07-11 09:21
artifacts/seed_mask_55

## CELL B — Unclassified Artifacts Scan

Glob `DATA_ROOT` recursively for anything matching `*finetune*`, `*al_*`, `*strategy*`, `*multiscale*` that is NOT already in the KEEP, QUARANTINE, or INSPECT lists. These may be artifacts missed by the v1.0 inventory.

In [5]:
# CELL B — Unclassified artifacts scan (read-only)
import glob

# Build the set of all known paths (normalized, no trailing slash)
all_known = set()
for p in KEEP_PATHS + QUARANTINE_PATHS + INSPECT_PATHS:
    all_known.add(p.rstrip('/'))

# Patterns that might match v1.0 fine-tuning artifacts
patterns = ['*finetune*', '*al_*', '*strategy*', '*multiscale*']
hits = set()
for pat in patterns:
    for match in glob.glob(os.path.join(DATA_ROOT, '**', pat), recursive=True):
        rel = os.path.relpath(match, DATA_ROOT).replace('\\', '/')
        # Skip anything inside known directories (e.g. individual .pt files inside
        # data/processed_finetune/35x35/ which are KEEP)
        skip = False
        for known in all_known:
            if rel.startswith(known.rstrip('/') + '/'):
                skip = True
                break
        if rel.rstrip('/') not in all_known and not skip:
            # Also skip anything inside prenorm_archive to avoid noise
            if 'prenorm_archive' not in rel:
                hits.add(rel)

print('UNCLASSIFIED — review manually')
print('=' * 80)
if hits:
    for h in sorted(hits):
        exists, size, mtime = path_info(DATA_ROOT, h)
        print(f'  {h:<60} {size:<20} {mtime}')
else:
    print('  (none found)')
print()

UNCLASSIFIED — review manually
  artifacts/chosen_transfer_strategy_results.json              0.00 MB              2026-07-25 06:37
  checkpoints/finetune_strategy_A.pt                           2.38 MB              2026-07-25 05:47
  checkpoints/finetune_strategy_B.pt                           2.38 MB              2026-07-25 06:02
  checkpoints/finetune_strategy_C.pt                           2.38 MB              2026-07-25 06:18
  checkpoints/finetune_strategy_D.pt                           2.38 MB              2026-07-25 06:28
  data/processed_finetune                                      14964 .pt / 14967 total files 2026-07-24 08:30
  figures/chunk-10 figures/counterfactual_ablation.png         0.14 MB              2026-07-21 01:32
  figures/chunk-10 figures/counterfactual_ablation_curve_distance.png 0.19 MB              2026-07-21 01:32
  figures/chunk-10 figures/counterfactual_ablation_paired.png  0.51 MB              2026-07-21 01:32
  splits/finetune_pool_indices_legacy_indepe

---
# PHASE 1 — Verify the Survivors BEFORE Moving Anything

Validate all KEEP paths while v1.0 artifacts are still in place, so that if a survivor is already broken the comparison context is still available.

## CELL C — Statistics & Checkpoint Verification

Verify `s11_mean.npy`, `s11_std.npy`, and `best_model.pt` are intact and match the expected Chunk 7 architecture (587,001 parameters).

In [6]:
# CELL C — Statistics and checkpoint verification
import numpy as np
import torch
from model import AntennaGNN

# ── s11_mean / s11_std ──
s11_mean = np.load(f'{DATA_ROOT}/artifacts/s11_mean.npy')
s11_std  = np.load(f'{DATA_ROOT}/artifacts/s11_std.npy')

assert s11_mean.shape == (201,), f'Expected s11_mean shape (201,), got {s11_mean.shape}'
assert s11_std.shape  == (201,), f'Expected s11_std shape (201,), got {s11_std.shape}'
assert s11_std.min() > 0, f'Expected s11_std.min() > 0, got {s11_std.min()}'

print('s11_mean:  mean={:.4f}  min={:.4f}  max={:.4f}'.format(
    s11_mean.mean(), s11_mean.min(), s11_mean.max()))
print('s11_std:   mean={:.4f}  min={:.4f}  max={:.4f}'.format(
    s11_std.mean(), s11_std.min(), s11_std.max()))

# ── best_model.pt ──
ckpt = torch.load(f'{DATA_ROOT}/checkpoints/best_model.pt',
                   map_location='cpu', weights_only=False)
assert 'model_state' in ckpt, f'Missing key "model_state"; keys found: {list(ckpt.keys())}'

model = AntennaGNN()
model.load_state_dict(ckpt['model_state'], strict=True)

total_params = sum(p.numel() for p in model.parameters())
print(f'\nbest_model.pt loaded successfully.')
print(f'  Total parameters: {total_params:,}')

assert total_params == 587001, (
    f'STOP — expected 587,001 parameters (Chunk 7 architecture), '
    f'got {total_params:,}. This is NOT the pretrained checkpoint '
    f'this project was built on.'
)
print('  ✓ Parameter count matches Chunk 7 architecture (587,001)')
del ckpt, model  # free memory

s11_mean:  mean=-1.2196  min=-5.6193  max=-0.0436
s11_std:   mean=1.3767  min=0.0139  max=7.1613

best_model.pt loaded successfully.
  Total parameters: 587,001
  ✓ Parameter count matches Chunk 7 architecture (587,001)


## CELL D — Processed Graph Spot Checks

For each grid size (35, 45, 55), sample 5 random `.pt` files from `data/processed_finetune/` and verify feature shapes, value ranges, and metadata. Values in the hundreds would indicate corrupted `.pt` files.

In [8]:
# CELL D — Processed graph spot checks (CORRECTED)
# Non-functioning antennas legitimately have min(S11) approaching 0 dB — they
# reflect nearly all power. Well-matched ones can go below -60 dB. So no tight
# range assertion is valid here. Structural checks are asserted; value plausibility
# is REPORTED; correctness is proven by direct comparison against the source .mat.
import torch, glob, random, os
import numpy as np
import pandas as pd
import scipy.io as sio

random.seed(42)
manifest = pd.read_csv(f'{DATA_ROOT}/artifacts/finetune_manifest.csv')
mf_idx = manifest.set_index(['grid_size', 'sample_idx'])
phase0_graph_checks = {}

for N in [35, 45, 55]:
    print(f"\n{'='*66}\n  Grid {N}x{N}\n{'='*66}")
    pt_dir = f'{DATA_ROOT}/data/processed_finetune/{N}x{N}'
    pt_files = sorted(glob.glob(os.path.join(pt_dir, 'sample_*.pt')))
    print(f"  Total .pt files: {len(pt_files)}")

    # Stratified sample: 5 functioning + 5 non-functioning
    sub = manifest[manifest['grid_size'] == N]
    func_idx = sub[sub['is_functioning'] == 1]['sample_idx'].tolist()
    nonf_idx = sub[sub['is_functioning'] == 0]['sample_idx'].tolist()
    picks = ([(i, 1) for i in random.sample(func_idx, min(5, len(func_idx)))] +
             [(i, 0) for i in random.sample(nonf_idx, min(5, len(nonf_idx)))])

    rows = []
    for idx, expect_func in picks:
        d = torch.load(f'{pt_dir}/sample_{idx}.pt', map_location='cpu',
                       weights_only=False)

        # --- structural: assert ---
        assert d.x.shape[1] == 5
        assert d.edge_attr.shape[1] == 2
        assert d.y.shape == (1, 201)
        assert d.grid_size == N
        assert abs(d.pixel_size_mm - 32.375 / N) < 1e-6
        assert hasattr(d, 'is_functioning')
        assert int(d.is_functioning) == expect_func, \
            f"sample_{idx}: is_functioning disagrees with manifest"

        y = d.y.flatten().numpy()
        # --- physics: S11 of a passive antenna cannot exceed 0 dB ---
        assert y.max() <= 0.01, f"sample_{idx}: y.max()={y.max():.3f} > 0 dB"
        # --- loose corruption tripwire only ---
        assert -200.0 < y.min() < 0.0, f"sample_{idx}: y.min()={y.min():.2f} implausible"

        rows.append({'idx': idx, 'func': expect_func,
                     'y_min': float(y.min()), 'y_max': float(y.max())})

    df = pd.DataFrame(rows)
    for f in [1, 0]:
        s = df[df['func'] == f]['y_min']
        label = 'functioning    ' if f else 'non-functioning'
        print(f"  {label}: y_min min={s.min():>8.2f}  median={s.median():>8.2f}  "
              f"max={s.max():>8.2f} dB   (n={len(s)})")
    print(f"  y_max across all sampled: max={df['y_max'].max():.3f} dB "
          f"(must be <= 0)")

    # --- REPORTED, not asserted: the -10 dB / is_functioning relationship.
    #     Chunk 14 Cell J measures this properly; asserting it here is circular.
    fr = (df[df['func'] == 1]['y_min'] < -10).mean()
    nr = (df[df['func'] == 0]['y_min'] > -10).mean()
    print(f"  functioning with y_min < -10 dB: {fr:.0%}   "
          f"non-functioning with y_min > -10 dB: {nr:.0%}")
    if fr < 0.8 or nr < 0.8:
        print("  [NOTE] weak agreement with a -10 dB threshold — expected if the "
              "dataset's resonant_freqs criterion differs; not an error here.")

    # --- DEFINITIVE: .pt must equal the source .mat ---
    raw_files = sorted(glob.glob(
        f'{RAW_DATA}/fine-tuning dataset/{N}x{N}/**/Mat_Files/*.mat',
        recursive=True))
    assert len(raw_files) == len(pt_files), \
        f"{N}x{N}: {len(raw_files)} .mat vs {len(pt_files)} .pt — ordering broken"
    for idx in [r['idx'] for r in rows[:3]]:
        mat = sio.loadmat(raw_files[idx])
        raw = mat['S11_dB'].flatten()
        d = torch.load(f'{pt_dir}/sample_{idx}.pt', map_location='cpu',
                       weights_only=False)
        assert np.allclose(d.y.flatten().numpy(), raw, atol=1e-4), \
            f"sample_{idx}: y does not match {os.path.basename(raw_files[idx])}"
        assert int(mat['resonant_freqs'].size > 0) == int(d.is_functioning)
    print(f"  ✓ 3 samples match their source .mat exactly (raw dB confirmed)")

    phase0_graph_checks[N] = {
        'total_pt_files': len(pt_files),
        'y_min_range': (float(df['y_min'].min()), float(df['y_min'].max())),
        'y_max_max': float(df['y_max'].max()),
    }

print("\n✓ Graph spot checks passed — .pt files hold raw dB, matching source .mat.")


  Grid 35x35
  Total .pt files: 4988
  functioning    : y_min min=  -21.37  median=  -13.25  max=  -10.67 dB   (n=5)
  non-functioning: y_min min=   -9.99  median=   -5.07  max=   -0.44 dB   (n=5)
  y_max across all sampled: max=-0.000 dB (must be <= 0)
  functioning with y_min < -10 dB: 100%   non-functioning with y_min > -10 dB: 100%
  ✓ 3 samples match their source .mat exactly (raw dB confirmed)

  Grid 45x45
  Total .pt files: 6984
  functioning    : y_min min=  -21.22  median=  -12.88  max=  -11.51 dB   (n=5)
  non-functioning: y_min min=   -8.38  median=   -1.36  max=   -0.16 dB   (n=5)
  y_max across all sampled: max=-0.000 dB (must be <= 0)
  functioning with y_min < -10 dB: 100%   non-functioning with y_min > -10 dB: 100%
  ✓ 3 samples match their source .mat exactly (raw dB confirmed)

  Grid 55x55
  Total .pt files: 2992
  functioning    : y_min min=  -18.72  median=  -13.22  max=  -11.14 dB   (n=5)
  non-functioning: y_min min=   -9.84  median=   -4.94  max=   -1.19 dB   

## CELL E — Splits and Manifest Verification

Load the three fine-tune split files and the manifest. Assert disjointness, complete coverage, and on-disk file existence. Report per-split, per-grid counts and functioning rates.

In [9]:
# CELL E — Splits and manifest verification
import json
import pandas as pd

# ── Load splits ──
with open(f'{DATA_ROOT}/splits/finetune_pool_indices.json') as f:
    pool_split = json.load(f)
with open(f'{DATA_ROOT}/splits/finetune_val_indices.json') as f:
    val_split = json.load(f)
with open(f'{DATA_ROOT}/splits/finetune_test_indices.json') as f:
    test_split = json.load(f)

# Convert to sets of (grid_size, sample_idx) tuples
pool_set = {(g, i) for g, i in pool_split}
val_set  = {(g, i) for g, i in val_split}
test_set = {(g, i) for g, i in test_split}

# ── Load manifest ──
manifest = pd.read_csv(f'{DATA_ROOT}/artifacts/finetune_manifest.csv')
manifest_set = {(int(row['grid_size']), int(row['sample_idx']))
                for _, row in manifest.iterrows()}

# ── Assert pairwise disjoint ──
assert len(pool_set & val_set)  == 0, f'pool ∩ val has {len(pool_set & val_set)} entries'
assert len(pool_set & test_set) == 0, f'pool ∩ test has {len(pool_set & test_set)} entries'
assert len(val_set & test_set)  == 0, f'val ∩ test has {len(val_set & test_set)} entries'
print('✓ Splits are pairwise disjoint')

# ── Assert union = manifest ──
union = pool_set | val_set | test_set
assert union == manifest_set, (
    f'Split union has {len(union)} entries, manifest has {len(manifest_set)}; '
    f'difference: {len(union.symmetric_difference(manifest_set))} entries'
)
print(f'✓ Split union has exactly {len(manifest_set)} entries = manifest size')

# ── Assert every entry exists on disk ──
missing_files = []
for g, i in union:
    fpath = f'{DATA_ROOT}/data/processed_finetune/{g}x{g}/sample_{i}.pt'
    if not os.path.exists(fpath):
        missing_files.append((g, i))

if missing_files:
    print(f'⚠ {len(missing_files)} split entries have no .pt file on disk!')
    for g, i in missing_files[:10]:
        print(f'  missing: {g}x{g}/sample_{i}.pt')
else:
    print(f'✓ All {len(union)} split entries exist as .pt files on disk')

# ── Per-split, per-grid counts ──
print(f'\n{"Split":<15} {"Grid":<8} {"Total":<8} {"Functioning":<12} {"Rate"}')
print('-' * 55)
for split_name, split_data in [('pool', pool_split), ('val', val_split), ('test', test_split)]:
    split_df = pd.DataFrame(split_data, columns=['grid_size', 'sample_idx'])
    for g in [35, 45, 55]:
        subset = split_df[split_df['grid_size'] == g]
        if len(subset) == 0:
            continue
        # Look up functioning status from manifest
        functioning = 0
        for _, row in subset.iterrows():
            mask = (manifest['grid_size'] == int(row['grid_size'])) & \
                   (manifest['sample_idx'] == int(row['sample_idx']))
            match = manifest[mask]
            if len(match) > 0 and match.iloc[0].get('is_functioning', 0) == 1:
                functioning += 1
        rate = functioning / len(subset) * 100 if len(subset) > 0 else 0
        print(f'{split_name:<15} {g:<8} {len(subset):<8} {functioning:<12} {rate:.1f}%')

# ── Manifest per-grid functioning rate ──
print(f'\nManifest per-grid functioning rate:')
for g in [35, 45, 55]:
    grid_data = manifest[manifest['grid_size'] == g]
    if len(grid_data) == 0:
        continue
    func_count = grid_data['is_functioning'].sum()
    rate = func_count / len(grid_data) * 100
    print(f'  {g}x{g}: {func_count}/{len(grid_data)} = {rate:.1f}%')
print('  (Expected approximately 63% / 65% / 48%)')

✓ Splits are pairwise disjoint
✓ Split union has exactly 14964 entries = manifest size
✓ All 14964 split entries exist as .pt files on disk

Split           Grid     Total    Functioning  Rate
-------------------------------------------------------
pool            35       3990     2530         63.4%
pool            45       5587     3647         65.3%
pool            55       2394     1159         48.4%
val             35       499      316          63.3%
val             45       698      456          65.3%
val             55       299      145          48.5%
test            35       499      317          63.5%
test            45       699      456          65.2%
test            55       299      144          48.2%

Manifest per-grid functioning rate:
  35x35: 3163/4988 = 63.4%
  45x45: 4559/6984 = 65.3%
  55x55: 1448/2992 = 48.4%
  (Expected approximately 63% / 65% / 48%)


## CELL F — Index Alignment Check

**This check has never previously been run.** Determine whether `indices.json` (Chunk 5) contains fine-tuning grid entries, and if so, verify that the numbering agrees between Chunk 5's `data/processed/` and Chunk 12's `data/processed_finetune/`. A mismatch here would mean the splits need rebuilding.

In [10]:
# CELL F — Index alignment check
import json
import torch
import glob
import random
import numpy as np
from scipy.io import loadmat

random.seed(42)

with open(f'{DATA_ROOT}/splits/indices.json') as f:
    indices = json.load(f)

# Check which keys contain fine-tuning grid sizes
ft_grids = {35, 45, 55}
ft_entries_by_key = {}
total_ft_entries = 0

for key in indices:
    entries = [(g, i) for g, i in indices[key] if g in ft_grids]
    if entries:
        ft_entries_by_key[key] = entries
        total_ft_entries += len(entries)

print(f'indices.json keys: {list(indices.keys())}')
print(f'Fine-tuning grid entries found: {total_ft_entries}')
for key, entries in ft_entries_by_key.items():
    grids = {}
    for g, i in entries:
        grids[g] = grids.get(g, 0) + 1
    print(f'  {key}: {dict(grids)}')

if total_ft_entries == 0:
    print('\nCase B — splits derived independently; no alignment risk')
    alignment_result = 'Case B — no fine-tuning entries in indices.json'
else:
    print(f'\nCase A — fine-tune splits may be adopted from Chunk 5')
    alignment_result = 'Case A — verifying alignment...'
    mismatches = []
    matches = 0

    for N in [35, 45, 55]:
        # Collect all (N, idx) from indices.json
        all_n_entries = []
        for key in indices:
            all_n_entries.extend([(g, i) for g, i in indices[key] if g == N])
        if not all_n_entries:
            continue

        c5_dir = f'{DATA_ROOT}/data/processed/{N}x{N}'
        c12_dir = f'{DATA_ROOT}/data/processed_finetune/{N}x{N}'

        if os.path.isdir(c5_dir) and len(glob.glob(os.path.join(c5_dir, 'sample_*.pt'))) > 0:
            # ── Direct comparison: load from both directories ──
            print(f'\n  {N}x{N}: Comparing data/processed/ vs data/processed_finetune/')
            sample_entries = random.sample(all_n_entries, min(10, len(all_n_entries)))

            for g, idx in sample_entries:
                c5_path  = f'{c5_dir}/sample_{idx}.pt'
                c12_path = f'{c12_dir}/sample_{idx}.pt'

                if not os.path.exists(c5_path) or not os.path.exists(c12_path):
                    print(f'    ⚠ sample_{idx}.pt: c5={os.path.exists(c5_path)}, c12={os.path.exists(c12_path)}')
                    mismatches.append((N, idx, 'file_missing'))
                    continue

                a = torch.load(c5_path, map_location='cpu', weights_only=False)
                b = torch.load(c12_path, map_location='cpu', weights_only=False)

                y_match = torch.allclose(a.y, b.y)
                x_match = torch.equal(a.x, b.x)

                if y_match and x_match:
                    matches += 1
                    print(f'    ✓ sample_{idx}.pt: y match, x match')
                else:
                    mismatches.append((N, idx, f'y_match={y_match}, x_match={x_match}'))
                    print(f'    ✗ sample_{idx}.pt: y_match={y_match}, x_match={x_match}')

        else:
            # ── Indirect comparison: re-derive from raw .mat files ──
            print(f'\n  {N}x{N}: data/processed/{N}x{N}/ not available — comparing .mat → .pt')
            raw_files = sorted(glob.glob(
                f'{RAW_DATA}/fine-tuning dataset/{N}x{N}/**/Mat_Files/*.mat',
                recursive=True))
            c12_pt_files = sorted(glob.glob(os.path.join(c12_dir, 'sample_*.pt')))

            print(f'    Raw .mat files: {len(raw_files)}')
            print(f'    Chunk 12 .pt files: {len(c12_pt_files)}')
            assert len(raw_files) == len(c12_pt_files), (
                f'STOP — {N}x{N}: {len(raw_files)} raw files vs '
                f'{len(c12_pt_files)} .pt files — cannot verify alignment'
            )

            sample_indices = random.sample(range(len(raw_files)), min(10, len(raw_files)))

            for idx in sample_indices:
                mat_path = raw_files[idx]
                pt_path = f'{c12_dir}/sample_{idx}.pt'
                if not os.path.exists(pt_path):
                    mismatches.append((N, idx, 'pt_file_missing'))
                    print(f'    ✗ sample_{idx}.pt: file missing')
                    continue

                mat = loadmat(mat_path)
                s11_mat = mat['S11_dB'].flatten().astype(np.float32)
                is_func_mat = int(mat['resonant_freqs'].size > 0)

                data = torch.load(pt_path, map_location='cpu', weights_only=False)
                s11_pt = data.y.squeeze().numpy()
                is_func_pt = int(data.is_functioning)

                s11_close = np.allclose(s11_mat, s11_pt, atol=1e-4)
                func_match = is_func_mat == is_func_pt

                if s11_close and func_match:
                    matches += 1
                    print(f'    ✓ sample_{idx}: S11 match, is_functioning match')
                else:
                    mismatches.append((N, idx, f's11_close={s11_close}, func={func_match}'))
                    print(f'    ✗ sample_{idx}: s11_close={s11_close}, func_match={func_match}')

    print(f'\n  Summary: {matches} matches, {len(mismatches)} mismatches')

    if mismatches:
        print('\n  ⛔ STOP — Index alignment mismatch detected!')
        print('  The splits would need rebuilding. Do NOT proceed to quarantine.')
        print('  Mismatching pairs:')
        for N, idx, reason in mismatches:
            print(f'    {N}x{N} sample_{idx}: {reason}')
        alignment_result = f'FAILED — {len(mismatches)} mismatches'
        raise AssertionError(f'Index alignment failed with {len(mismatches)} mismatches')
    else:
        alignment_result = f'PASSED — {matches} matches, 0 mismatches'
        print(f'\n  ✓ Index alignment verified: {matches} matches, 0 mismatches')

indices.json keys: ['train', 'val', 'test']
Fine-tuning grid entries found: 14964
  train: {45: 5587, 35: 3990, 55: 2394}
  val: {45: 698, 55: 299, 35: 499}
  test: {45: 699, 55: 299, 35: 499}

Case A — fine-tune splits may be adopted from Chunk 5

  35x35: Comparing data/processed/ vs data/processed_finetune/
    ✓ sample_4022.pt: y match, x match
    ✓ sample_470.pt: y match, x match
    ✓ sample_1565.pt: y match, x match
    ✓ sample_3855.pt: y match, x match
    ✓ sample_1218.pt: y match, x match
    ✓ sample_2179.pt: y match, x match
    ✓ sample_1590.pt: y match, x match
    ✓ sample_4574.pt: y match, x match
    ✓ sample_2434.pt: y match, x match
    ✓ sample_4145.pt: y match, x match

  45x45: Comparing data/processed/ vs data/processed_finetune/
    ✓ sample_3190.pt: y match, x match
    ✓ sample_5570.pt: y match, x match
    ✓ sample_1279.pt: y match, x match
    ✓ sample_3786.pt: y match, x match
    ✓ sample_4961.pt: y match, x match
    ✓ sample_3214.pt: y match, x match
 

## CELL G — Normalization-Statistics Leakage Check

**This check was defined in Chunk 12 Cell 15 but never run.** Determine whether any fine-tune test samples contributed to the computation of `s11_mean.npy` / `s11_std.npy`, which would constitute a mild train/test leak through normalization statistics.

In [11]:
# CELL G — Normalization-statistics leakage check
import json
import numpy as np

with open(f'{DATA_ROOT}/splits/indices.json') as f:
    indices = json.load(f)

# Chunk 5 training set entries that are fine-tuning grids
c5_ft_train = {(g, i) for g, i in indices.get('train', []) if g in [35, 45, 55]}
print(f'Chunk 5 train entries with fine-tuning grids: {len(c5_ft_train)}')

# Fine-tune test set
with open(f'{DATA_ROOT}/splits/finetune_test_indices.json') as f:
    ft_test = json.load(f)
ft_test_set = {(g, i) for g, i in ft_test}
print(f'Fine-tune test set size: {len(ft_test_set)}')

# Compute intersection
intersection = c5_ft_train & ft_test_set
print(f'\nIntersection (test samples that contributed to s11_mean/std): {len(intersection)}')

if intersection:
    share = len(intersection) / len(ft_test_set) * 100
    print(f'Share of fine-tune test set: {share:.1f}%')
    print(f'\n⚠ WARNING: {len(intersection)} fine-tune test samples were in the Chunk 5')
    print(f'  training set, meaning they contributed to s11_mean.npy / s11_std.npy.')
    print(f'  This is a MILD train/test leak through the normalization statistics.')
    print(f'  This count belongs in the paper\'s methods section.')
    leakage_result = f'{len(intersection)} samples ({share:.1f}% of test set)'
else:
    print('\n✓ No leakage — fine-tune test set is disjoint from Chunk 5 train set')
    leakage_result = 'No leakage'

# ── Check clean statistics ──
clean_mean_path = f'{DATA_ROOT}/artifacts/s11_mean_clean.npy'
clean_std_path  = f'{DATA_ROOT}/artifacts/s11_std_clean.npy'

if os.path.exists(clean_mean_path) and os.path.exists(clean_std_path):
    s11_mean_clean = np.load(clean_mean_path)
    s11_std_clean  = np.load(clean_std_path)
    s11_std_canon  = np.load(f'{DATA_ROOT}/artifacts/s11_std.npy')

    rel_diff = np.abs(s11_std_clean - s11_std_canon) / (np.abs(s11_std_canon) + 1e-10)
    max_rel_diff = rel_diff.max()

    print(f'\ns11_mean_clean.npy and s11_std_clean.npy exist.')
    print(f'  Max relative difference between clean and canonical std: {max_rel_diff:.6f}')
    print(f'\n  NOTE: The canonical (original) statistics remain the ones every chunk must')
    print(f'  use, because the pretrained head is calibrated to them. The clean arrays')
    print(f'  are a diagnostic only.')
else:
    print(f'\ns11_mean_clean.npy: {"exists" if os.path.exists(clean_mean_path) else "not found"}')
    print(f's11_std_clean.npy:  {"exists" if os.path.exists(clean_std_path) else "not found"}')
    print('  (Clean statistics not available — no relative comparison possible)')

Chunk 5 train entries with fine-tuning grids: 11971
Fine-tune test set size: 1497

Intersection (test samples that contributed to s11_mean/std): 0

✓ No leakage — fine-tune test set is disjoint from Chunk 5 train set

s11_mean_clean.npy and s11_std_clean.npy exist.
  Max relative difference between clean and canonical std: 0.046597

  NOTE: The canonical (original) statistics remain the ones every chunk must
  use, because the pretrained head is calibrated to them. The clean arrays
  are a diagnostic only.


---
# PHASE 2 — Archive (copy only, verify, then move)

Create the archive directory, copy all QUARANTINE paths, verify each copy with size + partial MD5 comparison, then move originals out of their live paths.

## CELL H — Copy & Verify QUARANTINE Paths into Archive

Create `DATA_ROOT/prenorm_archive_v1/` with subdirectories. If a partial `prenorm_archive/` already exists from an earlier manual attempt, merge its contents. Copy all QUARANTINE paths and verify every copy before proceeding.

In [12]:
# CELL H — Copy and verify QUARANTINE paths into archive
import shutil
import hashlib

ARCHIVE_ROOT = f'{DATA_ROOT}/prenorm_archive_v1'

# ── Create archive subdirectories ──
for subdir in ['artifacts', 'checkpoints', 'figures', 'artifacts/al_round_states']:
    os.makedirs(os.path.join(ARCHIVE_ROOT, subdir), exist_ok=True)
print(f'Archive root: {ARCHIVE_ROOT}')

# ── Check for a pre-existing prenorm_archive/ ──
old_archive = f'{DATA_ROOT}/prenorm_archive'
if os.path.isdir(old_archive):
    print(f'\nFound pre-existing {old_archive}/')
    for root, dirs, files in os.walk(old_archive):
        for fname in files:
            old_path = os.path.join(root, fname)
            rel = os.path.relpath(old_path, old_archive)
            new_path = os.path.join(ARCHIVE_ROOT, rel)
            if os.path.exists(new_path):
                print(f'  SKIP (already exists): {rel}')
            else:
                os.makedirs(os.path.dirname(new_path), exist_ok=True)
                shutil.copy2(old_path, new_path)
                print(f'  MERGED: {rel}')

def md5_partial(filepath, chunk_size=1024*1024):
    """MD5 of the first and last 1 MB of a file."""
    h = hashlib.md5()
    size = os.path.getsize(filepath)
    with open(filepath, 'rb') as f:
        h.update(f.read(min(chunk_size, size)))
        if size > chunk_size:
            f.seek(max(0, size - chunk_size))
            h.update(f.read(chunk_size))
    return h.hexdigest()

def count_files_recursive(dirpath):
    """Count all files recursively in a directory."""
    count = 0
    for root, dirs, files in os.walk(dirpath):
        count += len(files)
    return count

# ── Copy QUARANTINE paths ──
print(f'\nCopying QUARANTINE paths to archive...')
print(f'{"Source":<60} {"Status"}')
print('-' * 80)

copied_paths = []  # track what was actually copied for the verify step

for rel_path in QUARANTINE_PATHS:
    src = os.path.join(DATA_ROOT, rel_path)
    dst = os.path.join(ARCHIVE_ROOT, rel_path)

    if not os.path.exists(src):
        print(f'{rel_path:<60} SKIP (not present)')
        continue

    if os.path.isdir(src):
        if os.path.isdir(dst):
            # Merge — copy files that don't already exist
            for root, dirs, files in os.walk(src):
                for fname in files:
                    s = os.path.join(root, fname)
                    rel = os.path.relpath(s, src)
                    d = os.path.join(dst, rel)
                    if os.path.exists(d):
                        print(f'  {rel_path}/{rel:<53} SKIP (exists in archive)')
                    else:
                        os.makedirs(os.path.dirname(d), exist_ok=True)
                        shutil.copy2(s, d)
        else:
            shutil.copytree(src, dst)
        copied_paths.append((rel_path, 'dir'))
        print(f'{rel_path:<60} COPIED (directory)')
    else:
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if os.path.exists(dst):
            print(f'{rel_path:<60} SKIP (exists in archive)')
        else:
            shutil.copy2(src, dst)
        copied_paths.append((rel_path, 'file'))
        print(f'{rel_path:<60} COPIED')

# ── Verify each copy ──
print(f'\n{"=" * 80}')
print(f'  VERIFICATION')
print(f'{"=" * 80}')
print(f'{"Path":<60} {"Result"}')
print('-' * 80)

all_verified = True
verified_paths = []

for rel_path, ftype in copied_paths:
    src = os.path.join(DATA_ROOT, rel_path)
    dst = os.path.join(ARCHIVE_ROOT, rel_path)

    if ftype == 'dir':
        src_count = count_files_recursive(src)
        dst_count = count_files_recursive(dst)
        if src_count == dst_count:
            print(f'{rel_path:<60} PASS ({src_count} files)')
            verified_paths.append(rel_path)
        else:
            print(f'{rel_path:<60} FAIL (src={src_count}, dst={dst_count})')
            all_verified = False
    else:
        src_size = os.path.getsize(src)
        dst_size = os.path.getsize(dst)
        src_md5 = md5_partial(src)
        dst_md5 = md5_partial(dst)
        if src_size == dst_size and src_md5 == dst_md5:
            print(f'{rel_path:<60} PASS ({src_size / 1e6:.2f} MB, md5 match)')
            verified_paths.append(rel_path)
        else:
            print(f'{rel_path:<60} FAIL (size: {src_size} vs {dst_size}, md5: {src_md5} vs {dst_md5})')
            all_verified = False

if not all_verified:
    raise AssertionError(
        'STOP — verification failed for one or more copies. '
        'No files will be moved. Fix the issue and re-run.'
    )

print(f'\n✓ All {len(verified_paths)} copies verified successfully.')

Archive root: /content/drive/MyDrive/antenna_gnn/prenorm_archive_v1

Copying QUARANTINE paths to archive...
Source                                                       Status
--------------------------------------------------------------------------------
artifacts/transfer_strategy_comparison.csv                   COPIED
artifacts/chosen_transfer_strategy.json                      COPIED
artifacts/al_curves.csv                                      COPIED
artifacts/random_results.csv                                 SKIP (not present)
artifacts/al_composition.csv                                 SKIP (not present)
artifacts/final_test_results.json                            COPIED
artifacts/label_definition_reconciliation.json               COPIED
artifacts/al_round_states/                                   COPIED (directory)
checkpoints/strategy_A.pt                                    SKIP (not present)
checkpoints/strategy_B.pt                                    SKIP (not present)
che

## CELL I — Write Archive README

Write `prenorm_archive_v1/README.md` documenting the defect, headline v1.0 numbers, and an explicit warning against reuse.

In [13]:
# CELL I — Write archive README
from datetime import datetime

readme_content = f"""# prenorm_archive_v1 — Quarantined v1.0 Fine-Tuning Outputs

**Archived on:** {datetime.now().strftime('%Y-%m-%d %H:%M UTC')}

These are the v1.0 fine-tuning outputs produced under the normalization defect.

## The Defect

`FinetuneDataset` never z-scored the target S11 spectrum at load time, while
`evaluate()` applied de-normalization (multiplying by `s11_std` and adding
`s11_mean`). This means an inverse transform was applied to un-transformed data.
The consequences:

- Every dB value in these files is inflated by roughly a factor of `s11_std`
  (approximately 3.3).
- Resonance detection (S11 < -10 dB threshold) over-fires by about 22
  percentage points.
- Gradient clipping was saturating throughout training.

## Headline v1.0 Numbers (for recovery / comparison only)

| Metric | Value |
|--------|-------|
| Strategy selected | D (weighted val MAE 1.1875) |
| AL test S11 MAE | 1.2063 |
| Random test S11 MAE | 1.0977 |
| Zero-shot MAE range | 3.49 – 4.13 |
| find_peaks / manifest label agreement | 77.7% |

## ⚠ WARNING

**DO NOT** load anything in this directory into a v2.0 notebook. These files
are retained as a record of the defect only.

## Reference

See Part 0 of `antenna_gnn_tl_al_promptbook.md` v2.0 for the full defect
analysis and remediation plan.
"""

readme_path = os.path.join(ARCHIVE_ROOT, 'README.md')
with open(readme_path, 'w') as f:
    f.write(readme_content)

print(f'✓ Wrote {readme_path}')
print(f'  ({len(readme_content)} characters)')

✓ Wrote /content/drive/MyDrive/antenna_gnn/prenorm_archive_v1/README.md
  (1251 characters)


## CELL J — Move (Remove) Verified QUARANTINE Paths from Live Locations

Remove each verified QUARANTINE path from its live location. Only paths that passed verification in Cell H are touched. The archive copy in `prenorm_archive_v1/` is the permanent record.

In [14]:
# CELL J — Move (remove) verified QUARANTINE paths from live locations
import shutil

print('Removing verified QUARANTINE paths from live locations...')
print(f'{"Path":<60} {"Action"}')
print('-' * 80)

moved_count = 0

for rel_path in QUARANTINE_PATHS:
    live_path = os.path.join(DATA_ROOT, rel_path)

    if not os.path.exists(live_path):
        print(f'{rel_path:<60} SKIP (already absent)')
        continue

    # Only move if it was verified in Cell H
    if rel_path not in verified_paths:
        print(f'{rel_path:<60} SKIP (not verified — will not move)')
        continue

    if os.path.isdir(live_path):
        shutil.rmtree(live_path)
        print(f'{rel_path:<60} REMOVED (directory)')
    else:
        os.remove(live_path)
        print(f'{rel_path:<60} REMOVED')
    moved_count += 1

print(f'\n✓ {moved_count} QUARANTINE paths removed from live locations.')
print(f'  All copies preserved in {ARCHIVE_ROOT}/')

Removing verified QUARANTINE paths from live locations...
Path                                                         Action
--------------------------------------------------------------------------------
artifacts/transfer_strategy_comparison.csv                   REMOVED
artifacts/chosen_transfer_strategy.json                      REMOVED
artifacts/al_curves.csv                                      REMOVED
artifacts/random_results.csv                                 SKIP (already absent)
artifacts/al_composition.csv                                 SKIP (already absent)
artifacts/final_test_results.json                            REMOVED
artifacts/label_definition_reconciliation.json               REMOVED
artifacts/al_round_states/                                   REMOVED (directory)
checkpoints/strategy_A.pt                                    SKIP (already absent)
checkpoints/strategy_B.pt                                    SKIP (already absent)
checkpoints/strategy_C.pt          

---
# PHASE 3 — Clean-State Assertion

Verify the workspace is ready for v2.0 Chunk 12.

## CELL K — Post-Cleanup Workspace Assertions

Assert that every KEEP path still exists (with matching file counts), no QUARANTINE path is reachable at its live location, the archive is populated, and the checkpoint and graph spot checks still pass.

In [15]:
# CELL K — Post-cleanup workspace assertions
import torch
import glob
import random

random.seed(42)

failures = []

# ── 1. Every KEEP path still exists ──
print('Checking KEEP paths...')
for p in KEEP_PATHS:
    full = os.path.join(DATA_ROOT, p)
    if not os.path.exists(full):
        failures.append(f'KEEP path missing: {p}')
        print(f'  ✗ {p} — MISSING')
    else:
        # For directories, verify file counts match Phase 0
        if os.path.isdir(full):
            pt_count = len(glob.glob(os.path.join(full, 'sample_*.pt')))
            # Extract grid size from path
            for N in [35, 45, 55]:
                if f'{N}x{N}' in p and N in phase0_graph_checks:
                    expected = phase0_graph_checks[N]['total_pt_files']
                    if pt_count != expected:
                        failures.append(
                            f'KEEP dir {p}: expected {expected} .pt files, found {pt_count}')
                        print(f'  ✗ {p} — file count changed: {pt_count} (expected {expected})')
                    else:
                        print(f'  ✓ {p} ({pt_count} .pt files)')
                    break
            else:
                print(f'  ✓ {p} (directory exists)')
        else:
            print(f'  ✓ {p}')

# ── 2. No QUARANTINE path exists at live location ──
print('\nChecking QUARANTINE paths are gone...')
for p in QUARANTINE_PATHS:
    full = os.path.join(DATA_ROOT, p)
    if os.path.exists(full):
        failures.append(f'QUARANTINE path still exists: {p}')
        print(f'  ✗ {p} — STILL PRESENT')
    else:
        print(f'  ✓ {p} (removed)')

# ── 3. Archive exists and is populated ──
print('\nChecking archive...')
archive_file_count = count_files_recursive(ARCHIVE_ROOT)
if archive_file_count >= moved_count:
    print(f'  ✓ {ARCHIVE_ROOT}/ contains {archive_file_count} files (moved {moved_count})')
else:
    failures.append(
        f'Archive has {archive_file_count} files, expected at least {moved_count}')
    print(f'  ✗ Archive has {archive_file_count} files, expected >= {moved_count}')

# ── 4. best_model.pt loads and has 587001 parameters ──
print('\nChecking best_model.pt...')
try:
    ckpt = torch.load(f'{DATA_ROOT}/checkpoints/best_model.pt',
                       map_location='cpu', weights_only=False)
    model = AntennaGNN()
    model.load_state_dict(ckpt['model_state'], strict=True)
    total_params = sum(p.numel() for p in model.parameters())
    if total_params == 587001:
        print(f'  ✓ best_model.pt: {total_params:,} parameters')
    else:
        failures.append(f'best_model.pt has {total_params} params, expected 587001')
        print(f'  ✗ best_model.pt: {total_params:,} parameters (expected 587,001)')
    del ckpt, model
except Exception as e:
    failures.append(f'best_model.pt load error: {e}')
    print(f'  ✗ best_model.pt: {e}')

# ── 5. Graph spot checks still pass ──
print('\nRe-running graph spot checks...')
for N in [35, 45, 55]:
    pt_dir = f'{DATA_ROOT}/data/processed_finetune/{N}x{N}'
    pt_files = sorted(glob.glob(os.path.join(pt_dir, 'sample_*.pt')))
    if len(pt_files) < 2:
        failures.append(f'{N}x{N}: fewer than 2 .pt files')
        continue

    samples = random.sample(pt_files, min(3, len(pt_files)))
    for fpath in samples:
        fname = os.path.basename(fpath)
        try:
            data = torch.load(fpath, map_location='cpu', weights_only=False)
            assert data.x.shape[1] == 5
            assert data.y.shape == (1, 201)
            y_min = data.y.min().item()
            assert y_min < 0 and -60 < y_min < -1
            assert data.grid_size == N
            print(f'  ✓ {N}x{N}/{fname}: y.min()={y_min:.2f} dB')
        except Exception as e:
            failures.append(f'{N}x{N}/{fname}: {e}')
            print(f'  ✗ {N}x{N}/{fname}: {e}')

# ── Final verdict ──
print(f'\n{"=" * 80}')
if failures:
    print(f'  DO NOT PROCEED — {len(failures)} failure(s):')
    for i, f in enumerate(failures, 1):
        print(f'    {i}. {f}')
else:
    print('  WORKSPACE CLEAN — v2.0 Chunk 12 may proceed')
print(f'{"=" * 80}')

Checking KEEP paths...
  ✓ checkpoints/best_model.pt
  ✓ artifacts/s11_mean.npy
  ✓ artifacts/s11_std.npy
  ✓ artifacts/seed_mask_25.npy
  ✓ artifacts/seed_mask_35.npy
  ✓ artifacts/seed_mask_45.npy
  ✓ artifacts/seed_mask_55.npy
  ✓ artifacts/finetune_manifest.csv
  ✓ splits/indices.json
  ✓ splits/finetune_pool_indices.json
  ✓ splits/finetune_val_indices.json
  ✓ splits/finetune_test_indices.json
  ✓ data/processed_finetune/35x35/ (4988 .pt files)
  ✓ data/processed_finetune/45x45/ (6984 .pt files)
  ✓ data/processed_finetune/55x55/ (2992 .pt files)

Checking QUARANTINE paths are gone...
  ✓ artifacts/transfer_strategy_comparison.csv (removed)
  ✓ artifacts/chosen_transfer_strategy.json (removed)
  ✓ artifacts/al_curves.csv (removed)
  ✓ artifacts/random_results.csv (removed)
  ✓ artifacts/al_composition.csv (removed)
  ✓ artifacts/final_test_results.json (removed)
  ✓ artifacts/label_definition_reconciliation.json (removed)
  ✓ artifacts/al_round_states/ (removed)
  ✓ checkpoints/s

## CELL L — Remediation Summary & Report

Print and save a structured JSON report of the cleanup results to `DATA_ROOT/artifacts/v1_cleanup_report.json`.

In [16]:
# CELL L — Remediation summary and report
import json
from datetime import datetime

report = {
    'timestamp': datetime.now().isoformat(),
    'notebook': 'cleanup_v1_artifacts.ipynb',
    'files_archived': len(verified_paths),
    'files_moved': moved_count,
    'keep_paths_verified': len(KEEP_PATHS),
    'keep_paths_missing': keep_missing,
    'quarantine_paths_present_at_start': q_present,
    'quarantine_paths_remaining': sum(
        1 for p in QUARANTINE_PATHS
        if os.path.exists(os.path.join(DATA_ROOT, p))
    ),
    'index_alignment_result': alignment_result,
    'leakage_check_result': leakage_result,
    'archive_location': 'prenorm_archive_v1/',
    'post_cleanup_failures': failures if failures else [],
    'workspace_clean': len(failures) == 0,
}

print('=' * 60)
print('  V1.0 CLEANUP — REMEDIATION SUMMARY')
print('=' * 60)
for key, val in report.items():
    if isinstance(val, list):
        print(f'  {key}: [{len(val)} items]')
    else:
        print(f'  {key}: {val}')

report_path = f'{DATA_ROOT}/artifacts/v1_cleanup_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f'\n✓ Report saved to {report_path}')

  V1.0 CLEANUP — REMEDIATION SUMMARY
  timestamp: 2026-07-31T08:29:26.295556
  notebook: cleanup_v1_artifacts.ipynb
  files_archived: 11
  files_moved: 11
  keep_paths_verified: 15
  keep_paths_missing: 0
  quarantine_paths_present_at_start: 11
  quarantine_paths_remaining: 0
  index_alignment_result: PASSED — 30 matches, 0 mismatches
  leakage_check_result: No leakage
  archive_location: prenorm_archive_v1/
  post_cleanup_failures: [0 items]
  workspace_clean: True

✓ Report saved to /content/drive/MyDrive/antenna_gnn/artifacts/v1_cleanup_report.json
